# Location-Only SDM

This notebook trains a simple SDM-style baseline where the model only sees location `(lat, lon)` and predicts species composition.

It keeps the same species output space as the PseudoPA baseline, but removes species-set inputs from the encoder path during training and evaluation.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch
from torch.utils.data import DataLoader

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parents[1]
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from geoplant_pseudopa import (
    JointTrainingConfig,
    SpeciesSetDataset,
    evaluate_location_sdm,
    train_location_sdm_model,
)


In [ ]:
RAW_PA_TRAIN = REPO_ROOT / "dataset" / "data" / "PresenceAbsenceSurveys" / "PA_metadata_train.csv"
RAW_PA_TEST = REPO_ROOT / "dataset" / "data" / "PresenceAbsenceSurveys" / "PA_metadata_test.csv"
RAW_PA_TEST_LABELS = REPO_ROOT / "dataset" / "data" / "PresenceAbsenceSurveys" / "test_labels.csv"
RAW_PO_TRAIN = REPO_ROOT / "dataset" / "data" / "PresenceOnlyOccurrences" / "PO_metadata_train.csv"
PREPARED_DIR = NOTEBOOK_DIR / "outputs" / "prepared_metadata"
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

COUNTRY_FILTER = None
TEST_COUNTRY_FILTER = ["Denmark"]
VAL_FRACTION = 0.1
SPLIT_SEED = 42
NUM_SPECIES = 15286
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64

config = JointTrainingConfig(
    num_species=NUM_SPECIES,
    batch_size=BATCH_SIZE,
    warmup_pa_epochs=5,
    joint_epochs=20,
    use_location=True,
    mask_ratio=1.0,
    pa_negative_ratio=8.0,
    po_negative_ratio=4.0,
)

RAW_PA_TRAIN, RAW_PO_TRAIN, TEST_COUNTRY_FILTER, DEVICE

In [ ]:
def apply_country_filter(raw_frame: pd.DataFrame, country_filter):
    if country_filter is None:
        return raw_frame.reset_index(drop=True)
    if "country" in raw_frame.columns:
        return raw_frame[raw_frame["country"].isin(country_filter)].reset_index(drop=True)
    return raw_frame.reset_index(drop=True)


def aggregate_observations(raw_frame: pd.DataFrame, source_name: str) -> pd.DataFrame:
    aggregated = (
        raw_frame.groupby("surveyId")
        .agg(
            species_set=("speciesId", lambda col: "{" + ",".join(str(int(species_id)) for species_id in sorted(set(col.dropna().astype(int)))) + "}"),
            lat=("lat", "mean"),
            lon=("lon", "mean"),
        )
        .reset_index()
        .rename(columns={"surveyId": "survey_id"})
    )
    aggregated["survey_id"] = aggregated["survey_id"].astype(str)
    aggregated["source"] = source_name
    return aggregated


raw_pa_train = apply_country_filter(pd.read_csv(RAW_PA_TRAIN), COUNTRY_FILTER)
raw_po_train = apply_country_filter(pd.read_csv(RAW_PO_TRAIN), COUNTRY_FILTER)

prepared_pa = aggregate_observations(raw_pa_train, "PA")
prepared_po = aggregate_observations(raw_po_train, "PO")

unique_surveys = prepared_pa["survey_id"].sample(frac=1.0, random_state=SPLIT_SEED).reset_index(drop=True)
num_val = max(1, int(round(len(unique_surveys) * VAL_FRACTION)))
val_surveys = set(unique_surveys.iloc[:num_val])

prepared_pa["subset"] = prepared_pa["survey_id"].apply(lambda survey_id: "val" if survey_id in val_surveys else "train")
prepared_po["subset"] = "train"

pa_train_frame = prepared_pa[prepared_pa["subset"] == "train"].reset_index(drop=True)
pa_val_frame = prepared_pa[prepared_pa["subset"] == "val"].reset_index(drop=True)
po_train_frame = prepared_po.reset_index(drop=True)

pd.DataFrame([
    {"split": "PA train", "plots": len(pa_train_frame)},
    {"split": "PO train", "plots": len(po_train_frame)},
    {"split": "PA val", "plots": len(pa_val_frame)},
])

In [ ]:
pa_train_dataset = SpeciesSetDataset(pa_train_frame, config.num_species)
po_train_dataset = SpeciesSetDataset(po_train_frame, config.num_species)
pa_val_dataset = SpeciesSetDataset(pa_val_frame, config.num_species)

pa_train_loader = DataLoader(pa_train_dataset, batch_size=config.batch_size, shuffle=True)
po_train_loader = DataLoader(po_train_dataset, batch_size=config.batch_size, shuffle=True)
pa_val_loader = DataLoader(pa_val_dataset, batch_size=config.batch_size, shuffle=False)

example_batch = next(iter(pa_train_loader))
print("batch x:", tuple(example_batch["x"].shape))
print("batch loc:", tuple(example_batch["loc"].shape))

In [ ]:
model, history = train_location_sdm_model(
    pa_train_loader,
    po_train_loader,
    pa_val_loader,
    config=config,
    device=DEVICE,
)

history_df = pd.DataFrame(history)
history_df

In [ ]:
evaluate_location_sdm(model, pa_val_loader, device=torch.device(DEVICE))

## Denmark PA Test

This evaluates the location-only SDM on Denmark PA test plots using only their coordinates as input.

In [ ]:
raw_pa_test = apply_country_filter(pd.read_csv(RAW_PA_TEST), TEST_COUNTRY_FILTER)
test_labels = pd.read_csv(RAW_PA_TEST_LABELS)
test_labels = test_labels.rename(columns={"surveyId": "survey_id", "predictions": "species_set"})
test_labels["survey_id"] = test_labels["survey_id"].astype(str)
test_labels["species_set"] = test_labels["species_set"].apply(
    lambda value: "{" + ",".join(token for token in str(value).split() if token) + "}"
)

test_frame = raw_pa_test.rename(columns={"surveyId": "survey_id"}).copy()
test_frame["survey_id"] = test_frame["survey_id"].astype(str)
test_frame = test_frame.merge(test_labels[["survey_id", "species_set"]], on="survey_id", how="inner")
test_frame = test_frame[["survey_id", "lat", "lon", "species_set"]].drop_duplicates().reset_index(drop=True)

test_dataset = SpeciesSetDataset(test_frame, config.num_species)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

pd.DataFrame([
    {"split": "PA test", "plots": len(test_frame), "countries": TEST_COUNTRY_FILTER}
])

In [ ]:
pd.Series(evaluate_location_sdm(model, test_loader, device=torch.device(DEVICE)), name="value")

In [ ]:
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
torch.save(
    {
        "state_dict": model.state_dict(),
        "config": config.__dict__,
        "history": history,
        "task": "location_only_sdm",
    },
    OUTPUT_DIR / "location_only_sdm_model.pt",
)
history_df.to_csv(OUTPUT_DIR / "location_only_sdm_history.csv", index=False)
OUTPUT_DIR